In [0]:
#TATA Que

from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import col, explode
 
spark = SparkSession.builder.getOrCreate()
 
# ---------------- Employee DataFrame ----------------
emp_schema = StructType([
    StructField("emp_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("dept_id", IntegerType(), True),
    StructField("job_id", StringType(), True),
    StructField("salary", IntegerType(), True)
])
 
emp_df = spark.createDataFrame([
    (101, "Amit", 10, "DEV", 90000),
    (102, "Ravi", 20, "QA", 70000)
], emp_schema)
 
 
# ---------------- Organization JSON DataFrame ----------------
org_schema = StructType([
    StructField("organization", StructType([
        StructField("departments", ArrayType(
            StructType([
                StructField("dept_id", IntegerType(), True),
                StructField("dept_name", StringType(), True),
                StructField("jobs", ArrayType(
                    StructType([
                        StructField("job_id", StringType(), True),
                        StructField("job_name", StringType(), True)
                    ])
                ), True)
            ])
        ), True)
    ]))
])
 
org_df = spark.read.schema(org_schema).json("organization.json")
 
 
# ---------------- Flatten Organization Data ----------------
dept_df = org_df.select(explode("organization.departments").alias("dept"))\
    .select(
        col("dept.dept_id").alias("dept_id"),
        col("dept.dept_name").alias("dept_name"),
        explode("dept.jobs").alias("job"))\
    .select("dept_id","dept_name",
    col("job.job_id").alias("job_id"),
    col("job.job_name").alias("job_name")
)
 
 
# ---------------- Left Join ----------------
result_df = emp_df.join(
    dept_df,
    on=["dept_id", "job_id"],
    how="left"
)
 
result_df.show()